<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb

conn = duckdb.connect()
hf_key = userdata.get('HF_KEY')

conn.execute(f'CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN {hf_key})')

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients' : f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content' : f"read_parquet('{REL}/dim_content.parquet')",
    'fact_content_daily_performance' : f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_content_query_90d' : f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

DAILY_FILES = [
    f"{REL}/fact_content_daily_performance/month=2026-02/data_0.parquet",
    f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet",
    f"{REL}/fact_content_daily_performance/month=2026-04/data_0.parquet"
]

daily_source = f"""
read_parquet([
    '{DAILY_FILES[0]}',
    '{DAILY_FILES[1]}',
    '{DAILY_FILES[2]}'
])
"""


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page for one client. The feature window is from February 1, 2026 to April 30, 2026, with April 30 acting as the cutoff date. The target/label window is from May 1, 2026 to May 31, 2026.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

From fact_content_query_90d:
- impressions_90d
- clicks_90d
- avg_position_90d
- content_visible_query_count
- rare_query_count
- rare_impressions_share
- anonymized_impressions_share

From fact_content_daily_performance:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions
- ga4_engaged_sessions
- ga4_total_engagement_sec
- sessions_organic
- sessions_direct
- sessions_referral
- sessions_social
- sessions_paid
- sessions_ai

From dim_content:
- content_age at the April 30 cutoff
- freshness / days since last update at the April 30 cutoff


Label / Proxy:

The target is whether a page experiences a meaningful decline during the May 1–May 31, 2026 target window.

The starter proxy is is_declining_label, where trend_direction == "down". This is treated as a proxy rather than the final future-outcome label.


Context:

- client_hash_id
- content_hash_id
- query_hash_id
- window_start
- window_end
- report_date

These fields are used for identifying, joining, grouping, and controlling time windows rather than as predictive features.


Excluded / Leakage:

Target-window performance metrics and any information that was not available by the April 30, 2026 decision point are excluded from the feature set to prevent data leakage.

Identifiers and window/date metadata are also excluded from the model inputs.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ------------------------------------------------------------
# 1. Verify the grain of the daily performance table
#    Expected grain:
#    one row = one client + one content/page + one day
# ------------------------------------------------------------

daily_grain = conn.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        report_date
    )) AS unique_client_content_days

FROM {daily_source}
""").df()

display(daily_grain)


# ------------------------------------------------------------
# 2. Verify the feature-window dates
#    Expected:
#    earliest date = 2026-02-01
#    latest date   = 2026-04-30
# ------------------------------------------------------------

daily_window = conn.sql(f"""
SELECT
    MIN(report_date) AS earliest_date,
    MAX(report_date) AS latest_date,
    COUNT(DISTINCT report_date) AS number_of_days

FROM {daily_source}
""").df()

display(daily_window)


# ------------------------------------------------------------
# 3. Count the total number of daily observations
#    This tells us how many client-page-day rows
#    exist in the February-April feature window.
# ------------------------------------------------------------

daily_count = conn.sql(f"""
SELECT
    COUNT(*) AS total_daily_rows

FROM {daily_source}
""").df()

display(daily_count)


# ------------------------------------------------------------
# 4. Check missing values in important daily performance fields
#    Missing does NOT automatically mean zero.
# ------------------------------------------------------------

daily_missing = conn.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(gsc_impressions)
        AS missing_gsc_impressions,

    COUNT(*) - COUNT(gsc_clicks)
        AS missing_gsc_clicks,

    COUNT(*) - COUNT(gsc_avg_position)
        AS missing_gsc_avg_position,

    COUNT(*) - COUNT(ga4_pageviews)
        AS missing_ga4_pageviews,

    COUNT(*) - COUNT(ga4_sessions)
        AS missing_ga4_sessions,

    COUNT(*) - COUNT(ga4_engaged_sessions)
        AS missing_ga4_engaged_sessions,

    COUNT(*) - COUNT(sessions_ai)
        AS missing_sessions_ai

FROM {daily_source}
""").df()

display(daily_missing)


# ------------------------------------------------------------
# 5. Verify the grain of fact_content_query_90d
#    Expected grain:
#    one row = one client + one content/page + one query
# ------------------------------------------------------------

query_grain = conn.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id,
        query_hash_id
    )) AS unique_client_content_queries

FROM {TABLES['fact_content_query_90d']}
""").df()

display(query_grain)


# ------------------------------------------------------------
# 6. Check the actual 90-day windows in fact_content_query_90d
#    This verifies what dates the existing 90-day features
#    actually represent instead of assuming the window.
# ------------------------------------------------------------

query_windows = conn.sql(f"""
SELECT
    MIN(window_start) AS earliest_window_start,
    MAX(window_start) AS latest_window_start,

    MIN(window_end) AS earliest_window_end,
    MAX(window_end) AS latest_window_end

FROM {TABLES['fact_content_query_90d']}
""").df()

display(query_windows)


# ------------------------------------------------------------
# 7. Show the different 90-day windows and their row counts
#    This is more detailed than the previous check and shows
#    how many different window_start/window_end combinations
#    actually exist.
# ------------------------------------------------------------

query_window_details = conn.sql(f"""
SELECT
    window_start,
    window_end,
    COUNT(*) AS row_count

FROM {TABLES['fact_content_query_90d']}

GROUP BY
    window_start,
    window_end

ORDER BY
    window_start,
    window_end
""").df()

display(query_window_details)


# ------------------------------------------------------------
# 8. Check missing values in important query-level features
# ------------------------------------------------------------

query_missing = conn.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(impressions_90d)
        AS missing_impressions_90d,

    COUNT(*) - COUNT(clicks_90d)
        AS missing_clicks_90d,

    COUNT(*) - COUNT(avg_position_90d)
        AS missing_avg_position_90d,

    COUNT(*) - COUNT(content_visible_query_count)
        AS missing_visible_query_count,

    COUNT(*) - COUNT(rare_query_count)
        AS missing_rare_query_count,

    COUNT(*) - COUNT(rare_impressions_share)
        AS missing_rare_impressions_share,

    COUNT(*) - COUNT(anonymized_impressions_share)
        AS missing_anonymized_impressions_share

FROM {TABLES['fact_content_query_90d']}
""").df()

display(query_missing)


# ------------------------------------------------------------
# 9. Verify the number of unique client-page pairs
#     This checks the grain we ultimately want for modeling:
#     one row = one page for one client.
# ------------------------------------------------------------

page_client_grain = conn.sql(f"""
SELECT
    COUNT(DISTINCT (
        client_hash_id,
        content_hash_id
    )) AS unique_client_content_pairs

FROM {daily_source}
""").df()

display(page_client_grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_days
0,27621216,27621216


,earliest_date,latest_date,number_of_days
0,2026-02-01,2026-04-30,89


,total_daily_rows
0,27621216


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30


,window_start,window_end,row_count
0,2026-04-02,2026-06-30,2414248


,total_rows,missing_impressions_90d,missing_clicks_90d,missing_avg_position_90d,missing_visible_query_count,missing_rare_query_count,missing_rare_impressions_share,missing_anonymized_impressions_share
0,2414248,0,0,0,0,0,0,0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,unique_client_content_pairs
0,380147


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The dataset has several limitations. Client histories are unbalanced, so different clients may have different amounts of historical data. Some early rows may contain GSC data without GA4 data because GA4 tracking started later for some clients, so missing GA4 values should not automatically be interpreted as zero traffic. The 90-day query windows may also overlap with our feature and target windows, so window dates must be checked carefully to avoid using information from after the April 30 cutoff. Therefore, features must be constructed only from information available by the cutoff date, and differences in data availability must be considered when comparing pages and clients.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.